# 7.4 完整训练模板

本 notebook 提供一个生产级的 PyTorch 训练模板，包括参数管理、日志系统、训练/验证循环、早停机制等

### 学习目标
- 掌握 argparse 进行超参数管理
- 学会配置 logging 日志系统
- 实现 AverageMeter 工具类
- 编写标准的 train_one_epoch() 和 evaluate() 函数
- 实现完整的训练循环和早停机制

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import argparse
import logging
import os
import time
import tempfile

print(f"PyTorch 版本: {torch.__version__}")

## 1. argparse：超参数管理

使用 `argparse` 管理超参数的好处：
- 命令行灵活调整参数
- 自动生成帮助文档
- 参数类型检查
- 默认值管理

In [ ]:
def parse_args(args_list=None):
    """解析命令行参数

    Args:
        args_list: 参数列表，None 时从命令行读取

    Returns:
        解析后的参数对象
    """
    parser = argparse.ArgumentParser(description="PyTorch 训练模板")

    # 数据参数
    parser.add_argument('--batch_size', type=int, default=32, help='批大小')
    parser.add_argument('--num_workers', type=int, default=2, help='数据加载线程数')

    # 模型参数
    parser.add_argument('--input_dim', type=int, default=20, help='输入维度')
    parser.add_argument('--hidden_dim', type=int, default=64, help='隐藏层维度')
    parser.add_argument('--output_dim', type=int, default=5, help='输出维度（类别数）')
    parser.add_argument('--dropout', type=float, default=0.3, help='Dropout 比率')

    # 训练参数
    parser.add_argument('--epochs', type=int, default=30, help='训练轮数')
    parser.add_argument('--lr', type=float, default=0.001, help='学习率')
    parser.add_argument('--weight_decay', type=float, default=1e-4, help='权重衰减')
    parser.add_argument('--patience', type=int, default=5, help='早停耐心值')

    # 其他参数
    parser.add_argument('--seed', type=int, default=42, help='随机种子')
    parser.add_argument('--save_dir', type=str, default='./outputs', help='保存目录')
    parser.add_argument('--resume', type=str, default=None, help='恢复训练的检查点路径')

    # 在 notebook 中使用 parse_known_args 避免与 Jupyter 参数冲突
    if args_list is not None:
        args, _ = parser.parse_known_args(args_list)
    else:
        args, _ = parser.parse_known_args()

    return args


# 在 notebook 中传入参数列表
args = parse_args(['--epochs', '20', '--lr', '0.001', '--batch_size', '16'])

print("超参数:")
for key, value in vars(args).items():
    print(f"  {key}: {value}")

## 2. Logging 日志系统

使用 `logging` 模块替代 `print`，支持同时输出到控制台和文件

In [ ]:
def setup_logger(log_dir: str, name: str = "train") -> logging.Logger:
    """配置日志系统

    Args:
        log_dir: 日志文件保存目录
        name: logger 名称

    Returns:
        配置好的 logger
    """
    os.makedirs(log_dir, exist_ok=True)

    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)

    # 清除已有的 handler（避免重复添加）
    logger.handlers.clear()

    # 格式
    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    # 控制台 handler
    console_handler = logging.StreamHandler()
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)

    # 文件 handler
    file_handler = logging.FileHandler(
        os.path.join(log_dir, f"{name}.log"),
        encoding='utf-8',
    )
    file_handler.setLevel(logging.INFO)
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

    return logger


# 创建 logger
log_dir = tempfile.mkdtemp()
logger = setup_logger(log_dir)
logger.info("日志系统初始化完成")
logger.info(f"日志保存到: {log_dir}")

## 3. AverageMeter 工具类

用于跟踪和计算训练过程中的平均值（损失、准确率等）

In [ ]:
class AverageMeter:
    """计算并存储平均值和当前值

    用于跟踪训练过程中的指标（loss, accuracy 等）
    """

    def __init__(self, name: str = "Metric"):
        self.name = name
        self.reset()

    def reset(self) -> None:
        """重置所有统计值"""
        self.val = 0.0     # 当前值
        self.avg = 0.0     # 平均值
        self.sum = 0.0     # 总和
        self.count = 0     # 样本数

    def update(self, val: float, n: int = 1) -> None:
        """更新统计值

        Args:
            val: 当前值
            n: 样本数量
        """
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

    def __str__(self) -> str:
        return f"{self.name}: {self.avg:.4f}"


# 使用示例
loss_meter = AverageMeter("Loss")
acc_meter = AverageMeter("Acc")

# 模拟几个 batch 的更新
for i in range(5):
    batch_loss = 1.0 - i * 0.15  # 模拟递减的损失
    batch_acc = 0.5 + i * 0.1    # 模拟递增的准确率
    batch_size = 32

    loss_meter.update(batch_loss, batch_size)
    acc_meter.update(batch_acc, batch_size)

    print(f"Batch {i+1}: loss={batch_loss:.4f}, acc={batch_acc:.4f}")

print(f"\n平均值: {loss_meter}, {acc_meter}")

## 4. 模型定义

In [ ]:
class ClassifierNet(nn.Module):
    """简单的分类网络"""

    def __init__(
        self,
        input_dim: int = 20,
        hidden_dim: int = 64,
        output_dim: int = 5,
        dropout: float = 0.3,
    ):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# 查看模型
model = ClassifierNet()
print(model)
print(f"参数总数: {sum(p.numel() for p in model.parameters()):,}")

## 5. train_one_epoch() 函数

标准的单 epoch 训练函数

In [ ]:
def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: torch.device,
    epoch: int,
    logger: logging.Logger,
) -> tuple:
    """训练一个 epoch

    Args:
        model: 模型
        dataloader: 训练数据加载器
        criterion: 损失函数
        optimizer: 优化器
        device: 计算设备
        epoch: 当前 epoch 编号
        logger: 日志记录器

    Returns:
        (平均损失, 平均准确率)
    """
    model.train()
    loss_meter = AverageMeter("Loss")
    acc_meter = AverageMeter("Acc")

    for batch_idx, (data, target) in enumerate(dataloader):
        # 数据迁移到设备
        data = data.to(device)
        target = target.to(device)

        # 前向传播
        output = model(data)
        loss = criterion(output, target)

        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # 计算准确率
        pred = output.argmax(dim=1)
        acc = (pred == target).float().mean()

        # 更新指标
        batch_size = data.size(0)
        loss_meter.update(loss.item(), batch_size)
        acc_meter.update(acc.item(), batch_size)

    logger.info(
        f"Train Epoch {epoch:3d} | "
        f"Loss: {loss_meter.avg:.4f} | "
        f"Acc: {acc_meter.avg:.4f}"
    )

    return loss_meter.avg, acc_meter.avg


print("train_one_epoch() 定义完成")

## 6. evaluate() 函数

使用 `@torch.no_grad()` 装饰器禁用梯度计算，节省内存

In [ ]:
@torch.no_grad()
def evaluate(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    logger: logging.Logger,
) -> tuple:
    """评估模型

    Args:
        model: 模型
        dataloader: 验证数据加载器
        criterion: 损失函数
        device: 计算设备
        logger: 日志记录器

    Returns:
        (平均损失, 平均准确率)
    """
    model.eval()
    loss_meter = AverageMeter("Loss")
    acc_meter = AverageMeter("Acc")

    for data, target in dataloader:
        data = data.to(device)
        target = target.to(device)

        output = model(data)
        loss = criterion(output, target)

        pred = output.argmax(dim=1)
        acc = (pred == target).float().mean()

        batch_size = data.size(0)
        loss_meter.update(loss.item(), batch_size)
        acc_meter.update(acc.item(), batch_size)

    logger.info(
        f"  Val        | "
        f"Loss: {loss_meter.avg:.4f} | "
        f"Acc: {acc_meter.avg:.4f}"
    )

    return loss_meter.avg, acc_meter.avg


print("evaluate() 定义完成")
print("\n注意: @torch.no_grad() 装饰器的作用:")
print("  - 禁用梯度计算")
print("  - 减少内存消耗")
print("  - 加速推理")

## 7. 早停机制

当验证集指标不再改善时，提前停止训练以防过拟合

In [ ]:
class EarlyStopping:
    """早停机制

    当验证指标在 patience 个 epoch 内没有改善时，停止训练
    """

    def __init__(
        self,
        patience: int = 5,
        min_delta: float = 0.0,
        mode: str = "min",
    ):
        """
        Args:
            patience: 容忍没有改善的 epoch 数
            min_delta: 最小改善量
            mode: 'min' 表示指标越小越好, 'max' 表示越大越好
        """
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.counter = 0
        self.best_value = None
        self.should_stop = False

    def __call__(self, value: float) -> bool:
        """检查是否应该停止

        Args:
            value: 当前的验证指标值

        Returns:
            True 表示应该停止
        """
        if self.best_value is None:
            self.best_value = value
            return False

        if self.mode == "min":
            improved = value < (self.best_value - self.min_delta)
        else:
            improved = value > (self.best_value + self.min_delta)

        if improved:
            self.best_value = value
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True

        return self.should_stop


# 演示早停
early_stopping = EarlyStopping(patience=3, mode='min')
losses = [1.0, 0.8, 0.6, 0.55, 0.56, 0.57, 0.58, 0.59]

for epoch, loss in enumerate(losses):
    stop = early_stopping(loss)
    status = "STOP" if stop else f"counter={early_stopping.counter}"
    print(f"Epoch {epoch+1}: loss={loss:.2f}, best={early_stopping.best_value:.2f}, {status}")

## 8. 创建合成数据集

In [ ]:
def create_synthetic_data(
    n_train: int = 500,
    n_val: int = 100,
    input_dim: int = 20,
    num_classes: int = 5,
    seed: int = 42,
) -> tuple:
    """创建合成数据集

    Args:
        n_train: 训练样本数
        n_val: 验证样本数
        input_dim: 输入维度
        num_classes: 类别数
        seed: 随机种子

    Returns:
        (训练 DataLoader, 验证 DataLoader)
    """
    torch.manual_seed(seed)

    # 生成有一定模式的数据
    X_train = torch.randn(n_train, input_dim)
    y_train = (X_train[:, :num_classes].argmax(dim=1)).long()

    X_val = torch.randn(n_val, input_dim)
    y_val = (X_val[:, :num_classes].argmax(dim=1)).long()

    train_dataset = TensorDataset(X_train, y_train)
    val_dataset = TensorDataset(X_val, y_val)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

    return train_loader, val_loader


train_loader, val_loader = create_synthetic_data()
print(f"训练集: {len(train_loader.dataset)} 样本, {len(train_loader)} 批次")
print(f"验证集: {len(val_loader.dataset)} 样本, {len(val_loader)} 批次")

## 9. 完整训练循环

将所有组件整合到一个完整的训练流程中

In [ ]:
def get_device() -> torch.device:
    """自动检测最佳计算设备"""
    if torch.cuda.is_available():
        return torch.device('cuda:0')
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


def save_checkpoint(
    state: dict,
    path: str,
) -> None:
    """保存检查点"""
    torch.save(state, path)


def main(args):
    """主训练函数

    Args:
        args: 命令行参数
    """
    # === 1. 初始化 ===
    save_dir = tempfile.mkdtemp()
    logger = setup_logger(save_dir, name="main_train")
    device = get_device()
    logger.info(f"设备: {device}")
    logger.info(f"保存目录: {save_dir}")

    # 设置随机种子
    torch.manual_seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)

    # === 2. 数据 ===
    train_loader, val_loader = create_synthetic_data(
        n_train=500,
        n_val=100,
        input_dim=args.input_dim,
        num_classes=args.output_dim,
        seed=args.seed,
    )
    logger.info(f"训练集: {len(train_loader.dataset)} 样本")
    logger.info(f"验证集: {len(val_loader.dataset)} 样本")

    # === 3. 模型 ===
    model = ClassifierNet(
        input_dim=args.input_dim,
        hidden_dim=args.hidden_dim,
        output_dim=args.output_dim,
        dropout=args.dropout,
    ).to(device)
    logger.info(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")

    # === 4. 优化器和调度器 ===
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        model.parameters(),
        lr=args.lr,
        weight_decay=args.weight_decay,
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=args.epochs,
    )

    # === 5. 早停 ===
    early_stopping = EarlyStopping(patience=args.patience, mode='min')
    best_val_loss = float('inf')
    start_epoch = 0

    # === 6. 恢复训练（如果有检查点） ===
    if args.resume and os.path.exists(args.resume):
        checkpoint = torch.load(args.resume, map_location=device, weights_only=True)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_val_loss = checkpoint['best_val_loss']
        logger.info(f"从 epoch {start_epoch} 恢复训练")

    # === 7. 训练循环 ===
    logger.info("=" * 60)
    logger.info("开始训练")
    logger.info("=" * 60)

    train_losses = []
    val_losses = []

    for epoch in range(start_epoch, args.epochs):
        start_time = time.time()

        # 训练
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device, epoch + 1, logger,
        )

        # 验证
        val_loss, val_acc = evaluate(
            model, val_loader, criterion, device, logger,
        )

        # 学习率调度
        scheduler.step()

        # 记录
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        elapsed = time.time() - start_time
        lr = optimizer.param_groups[0]['lr']
        logger.info(f"  Time: {elapsed:.1f}s | LR: {lr:.6f}")

        # 保存最佳模型
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_path = os.path.join(save_dir, "best_model.pth")
            torch.save(model.state_dict(), best_path)
            logger.info(f"  * 最佳模型已保存 (val_loss={val_loss:.4f})")

        # 保存检查点
        ckpt_path = os.path.join(save_dir, "checkpoint_latest.pth")
        save_checkpoint(
            {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_val_loss': best_val_loss,
                'train_loss': train_loss,
                'val_loss': val_loss,
            },
            ckpt_path,
        )

        # 早停检查
        if early_stopping(val_loss):
            logger.info(f"早停! 验证损失 {args.patience} 个 epoch 未改善")
            break

    # === 8. 训练完成 ===
    logger.info("=" * 60)
    logger.info(f"训练完成! 最佳验证损失: {best_val_loss:.4f}")
    logger.info("=" * 60)

    return model, train_losses, val_losses

In [ ]:
# 运行训练
args = parse_args([
    '--epochs', '20',
    '--lr', '0.001',
    '--patience', '7',
    '--hidden_dim', '64',
    '--dropout', '0.2',
])

model, train_losses, val_losses = main(args)

In [ ]:
# 可视化训练曲线
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(1, 1, figsize=(8, 4))
    ax.plot(train_losses, label='Train Loss', marker='o', markersize=3)
    ax.plot(val_losses, label='Val Loss', marker='s', markersize=3)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training and Validation Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib 未安装，跳过可视化")
    print(f"\n训练损失: {[f'{l:.4f}' for l in train_losses]}")
    print(f"验证损失: {[f'{l:.4f}' for l in val_losses]}")

## 10. 总结

### 训练模板结构
```
1. 解析参数 (argparse)
2. 设置日志 (logging)
3. 设置随机种子
4. 准备数据 (Dataset + DataLoader)
5. 创建模型 + 优化器 + 调度器
6. 训练循环:
   for epoch in range(epochs):
       train_one_epoch()    # 训练
       evaluate()           # 验证
       scheduler.step()     # 学习率调度
       save best model      # 保存最佳模型
       save checkpoint      # 保存检查点
       early stopping       # 早停检查
```

### 关键组件

| 组件 | 作用 |
|------|------|
| argparse | 命令行参数管理 |
| logging | 日志记录（控制台 + 文件） |
| AverageMeter | 跟踪指标平均值 |
| EarlyStopping | 防止过拟合 |
| Checkpoint | 断点续传 |

### 最佳实践
1. **始终设置随机种子** 保证可复现
2. **使用 logger 而不是 print** 便于记录和回溯
3. **评估时使用 @torch.no_grad()** 节省内存
4. **定期保存检查点** 防止意外中断
5. **使用早停** 防止过拟合
6. **使用 AverageMeter** 准确计算 epoch 指标

---

## 练习题

### 基础题
1. 在训练模板中添加一个 `--optimizer` 参数，支持选择 Adam 或 SGD
2. 修改 AverageMeter，添加一个 `history` 属性记录所有更新的值

### 进阶题
3. 修改训练模板，使其支持：
   - 多种学习率调度器（StepLR / CosineAnnealingLR / ReduceLROnPlateau）
   - 梯度裁剪（gradient clipping）
   - 混合精度训练（torch.cuda.amp）

### 挑战题
4. 将训练模板改写为一个可复用的 `Trainer` 类：
   - 支持自定义回调函数（on_epoch_start, on_epoch_end, on_train_end）
   - 支持多指标的早停和模型选择
   - 自动记录实验日志

完成本章学习后，请尝试 [exercises.md](./exercises.md) 中的更多练习题来巩固知识。